### DRAEM

In [ ]:
#configurations

train_image_path = '/home/maria/Documents/dataset_splited/agco_dataset_ablation/train/'
anomaly_source_path = '/home/maria/Documents/datasets/dtd/images/'
checkpoint_path = '/home/maria/Documents/projects/anomaly_detection_vehicles/checkpoints/'
run_name = 'scratch_noaug2_' #'TEST_DRAEM_10'
resize_image = (256, 256*2)
batch_size = 12
lr = 0.0001
epochs = 201

### Dataloader

In [ ]:
import torch

from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms

import torchvision
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader


In [ ]:
#dataset = CustomTrainDataset(image_dir=train_image_path, anomaly_source_dir=anomaly_source_path, resize_shape=resize_image)

#### Model - train

In [ ]:
#from models.DRAEM import ReconstructiveSubNetwork, DiscriminativeSubNetwork
#from losses.loss import FocalLoss, SSIM
from torch import optim
import os

In [ ]:
#lets test output,inout
model = ReconstructiveSubNetwork()
input_image = torch.randn(1, 3, 256, 256*2)
output_image = model(input_image)
print(output_image.shape)

In [ ]:
#lets test output,inout
model = DiscriminativeSubNetwork()
input_image = torch.randn(1, 3, 256, 256*2)
output_image = model(input_image)
print(output_image.shape)

In [ ]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        m.weight.data.normal_(0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        m.weight.data.normal_(1.0, 0.02)
        m.bias.data.fill_(0)

import matplotlib.pyplot as plt
import numpy as np



# Function to visualize a batch of images
def visualize_images(images, titles, colours, figsize=(15, 5)):
    """
    Display a list of images with titles.

    Args:
        images (list): List of images (tensors or numpy arrays) to display.
        titles (list): List of titles for each image.
        n_iter (int): Current iteration number.
        figsize (tuple): Figure size.
    """
    plt.figure(figsize=figsize)
    num_images = len(images)

    for i, (img, title, colour) in enumerate(zip(images, titles, colours)):
        plt.subplot(1, num_images, i + 1)

        # Detach the tensor, move to CPU, and convert to NumPy
        img_np = img.detach().cpu().numpy().transpose(1, 2, 0)  # Convert (C, H, W) to (H, W, C)

        if colour is None:
            plt.imshow(np.clip(img_np, 0, 1))
        else:
          plt.imshow(np.clip(img_np, 0, 1), cmap=colour)
        plt.title(f"{title}", fontsize=11)
        plt.axis("off")

    plt.show()



In [ ]:
def save_images(images, titles, flags, n_iter, epoch, run_name, figsize=(10, 10)):
    """
    Save a list of images with titles to a folder named after the epoch.

    Args:
        images (list): List of images (tensors or numpy arrays) to display.
        titles (list): List of titles for each image.
        flags (list): List of flags (e.g., colormap) for each image.
        n_iter (int): Current iteration number.
        epoch (int): Current epoch number.
        run_name (str): Name of the run for folder creation.
        figsize (tuple): Figure size for each image.
    """
    # Create the directory structure: run_name/visualisations/epoch/
    output_dir = os.path.join(run_name, "visualisations", str(epoch))
    os.makedirs(output_dir, exist_ok=True)

    for i, (img, title, flag) in enumerate(zip(images, titles, flags)):
        plt.figure(figsize=figsize)

        # Detach the tensor, move to CPU, and convert to NumPy
        img_np = img.detach().cpu().numpy().transpose(1, 2, 0)  # Convert (C, H, W) to (H, W, C)

        # Display the image
        if flag is None:
            plt.imshow(np.clip(img_np, 0, 1))
        else:
            plt.imshow(np.clip(img_np, 0, 1), cmap=flag)

        #plt.title(f"{title} (Iter {n_iter})")
        plt.axis("off")

        # Save the image with a descriptive filename
        filename = os.path.join(output_dir, f"{n_iter}_{title.replace(' ', '_')}.png")
        plt.savefig(filename, bbox_inches='tight')
        plt.close()

In [ ]:
%%capture
model = ReconstructiveSubNetwork(in_channels=3, out_channels=3)
model.apply(weights_init)
model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/project/DRAEM/checkpoints/scratch_noaug2_epoch_6.pckl"))
model.cuda()
#model.eval()

model_seg = DiscriminativeSubNetwork(in_channels=6, out_channels=2)
model_seg.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/project/DRAEM/checkpoints/scratch_noaug2_epoch_6_seg.pckl"))
model_seg.cuda()
#model_seg.eval()

In [ ]:
%%capture
model = ReconstructiveSubNetwork(in_channels=3, out_channels=3)
model.cuda()
model.apply(weights_init)

model_seg = DiscriminativeSubNetwork(in_channels=6, out_channels=2)
model_seg.cuda()
model_seg.apply(weights_init)

In [ ]:
optimizer = torch.optim.Adam([
                                      {"params": model.parameters(), "lr": lr},
                                      {"params": model_seg.parameters(), "lr": lr}])

scheduler = optim.lr_scheduler.MultiStepLR(optimizer,[epochs*0.8,epochs*0.9],gamma=0.2, last_epoch=-1)

loss_l2 = torch.nn.modules.loss.MSELoss()
loss_ssim = SSIM()
loss_focal = FocalLoss()

In [ ]:
dataset = CustomTrainDataset(image_dir=train_image_path, anomaly_source_dir=anomaly_source_path, resize_shape=resize_image)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=6)

In [ ]:
epochs = 100

In [ ]:
for epoch in range(7, epochs):
    n_iter = 0
    print("Epoch: "+str(epoch))
    for i_batch, sample_batched in enumerate(dataloader):

        gray_batch = sample_batched["image"].cuda()
        aug_gray_batch = sample_batched["augmented_image"].cuda()
        anomaly_mask = sample_batched["anomaly_mask"].cuda()

        gray_rec = model(aug_gray_batch)
        joined_in = torch.cat((gray_rec, aug_gray_batch), dim=1)

        out_mask = model_seg(joined_in)
        out_mask_sm = torch.softmax(out_mask, dim=1)

        l2_loss = loss_l2(gray_rec,gray_batch)
        ssim_loss = loss_ssim(gray_rec, gray_batch)

        segment_loss = loss_focal(out_mask_sm, anomaly_mask)
        loss = l2_loss + ssim_loss + segment_loss

        optimizer.zero_grad()

        loss.backward()
        optimizer.step()

        #if n_iter % 200 == 0:
        print(f"Iteration {n_iter} - L2 Loss: {l2_loss.item():.4f} | SSIM Loss: {ssim_loss.item():.4f} | Segment Loss: {segment_loss.item():.4f} | Total Loss: {loss.item():.4f}")

        if n_iter%100 == 0 or n_iter==2019:
            images = [aug_gray_batch[0], gray_batch[0], gray_rec[0], anomaly_mask[0], out_mask_sm[:, 1:, :, :][0]]
            titles = ["wo anomaly", "original augmented", "reconstructed", "original mask", "predicted mask"]
            flags = [None, None, None, 'grey', None]
            run_name = run_name

            save_images(images, titles, flags, n_iter, epoch, run_name)

        t_mask = out_mask_sm[:, 1:, :, :]

        binary_t_mask = (t_mask > 0.5).float()

        #if n_iter%100 == 0 or n_iter==2019:
        visualize_images(
            images=[aug_gray_batch[0], gray_batch[0], gray_rec[0], anomaly_mask[0],  t_mask[0], binary_t_mask[0]],
            titles=["wo anomaly", "original augmented", "reconstructed", "original mask", "predicted mask", "predicted mask"],
            colours=[None, None, None, 'grey', 'jet', 'grey']
        )

        n_iter +=1

    scheduler.step()
    torch.save(model.state_dict(), os.path.join(checkpoint_path, run_name+f"epoch_{epoch}.pckl"))
    torch.save(model_seg.state_dict(), os.path.join(checkpoint_path, run_name+f"epoch_{epoch}_seg.pckl"))